In [2]:
from azureml.core import Workspace

# Loads workspace details from config.json
ws = Workspace.from_config()



In [3]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = "cpu-cluster"

try:
    compute_target = ComputeTarget(workspace=ws, name=cluster_name)
    print(f"Found existing compute target: {cluster_name}")

except ComputeTargetException:
    print("Creating a new compute cluster...")

    compute_config = AmlCompute.provisioning_configuration(
        vm_size="Standard_D2s_v3",
        min_nodes=0,
        max_nodes=4
    )

    compute_target = ComputeTarget.create(
        ws,
        cluster_name,
        compute_config
    )

    compute_target.wait_for_completion(show_output=True)

print(compute_target.get_status().serialize())


Creating a new compute cluster...
InProgress....
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned
{'currentNodeCount': 0, 'targetNodeCount': 0, 'nodeStateCounts': {'preparingNodeCount': 0, 'runningNodeCount': 0, 'idleNodeCount': 0, 'unusableNodeCount': 0, 'leavingNodeCount': 0, 'preemptedNodeCount': 0}, 'allocationState': 'Steady', 'allocationStateTransitionTime': '2026-07-12T06:16:11.241000+00:00', 'errors': None, 'creationTime': '2026-07-12T06:15:55.014369+00:00', 'modifiedTime': '2026-07-12T06:16:12.632607+00:00', 'provisioningState': 'Succeeded', 'provisioningStateTransitionTime': None, 'scaleSettings': {'minNodeCount': 0, 'maxNodeCount': 4, 'nodeIdleTimeBeforeScaleDown': 'PT1800S'}, 'vmPriority': 'Dedicated', 'vmSize': 'Standard_D2s_v3'}


## Create an Azure ML Experiment

In [9]:
from azureml.core import Experiment 

experiment_name = "udacity-hyperdrive" 

exp = Experiment(
    workspace = ws,
    name = experiment_name
) 
print(f"Experiment '{experiment_name}' is ready.")

Experiment 'udacity-hyperdrive' is ready.


## Configure the Training Environment

In [10]:
from azureml.core import Environment 
from azureml.core.conda_dependencies import CondaDependencies 

# Create a custom environment
env = Environment(name = "udacity-training-env") 

# Define dependencies 
deps = CondaDependencies() 

# Define dependencies 
deps.add_conda_package("scikit-learn") 
deps.add_conda_package("pandas") 
deps.add_conda_package("numpy") 

# Pip packages 
deps.add_pip_package("azureml-defaults") 

# Attach dependencies to the environment 
env.python.conda_dependencies = deps 

print(f"Environment '{env.name}' created successfully.")


Environment 'udacity-training-env' created successfully.


## Configure the Training Script

In [11]:
from azureml.core import ScriptRunConfig 

src = ScriptRunConfig(
    source_directory=".",
    script = "train.py",
    compute_target = compute_target,
    environment =env,
)

print("ScriptRunConfig created successfully.")

ScriptRunConfig created successfully.


## Submit the Baseline Training Run

In [12]:
run = exp.submit(src) 
print("Run ID:", run.id) 


Run ID: udacity-hyperdrive_1783790850_d56fd73b


In [18]:
print("Current Status:", run.get_status())

Current Status: Completed


In [17]:
details = run.get_details()

print(details["status"])
print(details.get("error", "No error information available"))

Completed
No error information available


In [19]:
import pandas as pd

df = pd.read_csv("bank.csv")

print(df.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']


## Retrieve the Baseline Metrics

In [20]:
metrics = run.get_metrics() 
print(metrics)

{'Regularization Strength': 1.0, 'Max Iterations': 100, 'Accuracy': 0.8020600089565607}


In [21]:
print(run.get_file_names())

['outputs/model.pkl', 'system_logs/cs_capability/cs-capability.log', 'system_logs/hosttools_capability/hosttools-capability.log', 'system_logs/lifecycler/execution-wrapper.log', 'system_logs/lifecycler/lifecycler.log', 'system_logs/metrics_capability/metrics-capability.log', 'system_logs/snapshot_capability/snapshot-capability.log', 'user_logs/std_log.txt']


In [22]:
print(f"Baseline Accuracy : {metrics['Accuracy']:.4f}")
print(f"C                 : {metrics['Regularization Strength']}")
print(f"Max Iterations    : {metrics['Max Iterations']}")

Baseline Accuracy : 0.8021
C                 : 1.0
Max Iterations    : 100


## HyperDrive Configuration

In this section, we configure HyperDrive to search for the best combination of hyperparameters for the Logistic Regression model.

The hyperparameters to be tuned are:

- Regularization Strength (C)
- Maximum Iterations (max_iter)

In [23]:
from azureml.train.hyperdrive import (
    RandomParameterSampling,
    choice,
    BanditPolicy,
    HyperDriveConfig,
    PrimaryMetricGoal
)

In [24]:
param_sampling = RandomParameterSampling(
    {
        "--C": choice(0.001, 0.01, 0.1, 1.0, 10.0),
        "--max_iter": choice(100, 150, 200)
    }
)

print("Hyperparameter search space created successfully.")

Hyperparameter search space created successfully.


In [25]:
early_termination_policy = BanditPolicy(
    slack_factor = 0.1,
    evaluation_interval=1,
    delay_evaluation=5
) 
print("Bandit policy configured successfully.")

Bandit policy configured successfully.


In [26]:
hyperdrive_config = HyperDriveConfig(
    run_config = src,
    hyperparameter_sampling=param_sampling,
    policy = early_termination_policy,
    primary_metric_name = 'Accuracy',
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs = 12,
    max_concurrent_runs=4,
)

In [27]:
hyperdrive_run = exp.submit(hyperdrive_config) 


In [39]:
import _collections_abc
from azureml.widgets import RunDetails

RunDetails(hyperdrive_run).show()

_HyperDriveWidget(widget_settings={'childWidgetDisplay': 'popup', 'send_telemetry': False, 'log_level': 'INFO'…

## HyperDrive RunDetails Widget

The `RunDetails` widget was added as requested in the project review.

The widget object was successfully created, and the notebook environment was verified to support interactive `ipywidgets`. However, in the current Microsoft Azure Machine Learning managed notebook environment, the Azure ML SDK v1 `RunDetails` widget did not render despite successful execution.

The HyperDrive experiment completed successfully, and all child runs, metrics, best hyperparameters, and outputs were verified through Azure Machine Learning Studio.

To provide the same execution evidence that would normally be available through the `RunDetails` widget, screenshots of the HyperDrive parent run, child runs, best child run, and evaluation metrics have been included in the project README and screenshots folder. These screenshots demonstrate the successful execution of all HyperDrive trials, the evaluated hyperparameter combinations, the selected best-performing run, and the corresponding performance metrics.


In [36]:
print("HyperDrive Run ID:", hyperdrive_run.id) 
print('Status:', hyperdrive_run.get_status())

HyperDrive Run ID: HD_73dec79b-1610-44a1-96d9-7b7281a5c6ad
Status: Completed


In [35]:
print(hyperdrive_run.get_status())

Completed


## Retrieve the Best HyperDrive Run 

In [40]:
best_run = hyperdrive_run.get_best_run_by_primary_metric() 

print("Best Run ID:") 
print(best_run.id)

Best Run ID:
HD_73dec79b-1610-44a1-96d9-7b7281a5c6ad_2


In [41]:
best_run.get_details()["runDefinition"]["arguments"]

['--C', '0.01', '--max_iter', '150']

In [42]:
best_metrics = best_run.get_metrics() 
print(best_metrics) 

{'Regularization Strength': 0.01, 'Accuracy': 0.8051948051948052, 'Max Iterations': 150}


In [49]:
print(f"Best Accuracy                : {best_metrics['Accuracy']:.4f}")
print(f"Regularization Strength (C)  : {best_metrics['Regularization Strength']}")
print(f"Maximum Iterations           : {best_metrics['Max Iterations']}")

Best Accuracy                : 0.8052
Regularization Strength (C)  : 0.01
Maximum Iterations           : 150


In [50]:
best_model = best_run.register_model(
    model_name="udacity-hyperdrive-best-model",
    model_path="outputs/model.pkl"
)

print("Model Name :", best_model.name)
print("Version    :", best_model.version)

Model Name : udacity-hyperdrive-best-model
Version    : 4


In [51]:
baseline_metrics = run.get_metrics()

print("========== MODEL COMPARISON ==========")
print(f"Baseline Accuracy    : {baseline_metrics['Accuracy']:.4f}")
print(f"HyperDrive Accuracy  : {best_metrics['Accuracy']:.4f}")

========== MODEL COMPARISON ==========
Baseline Accuracy    : 0.8021
HyperDrive Accuracy  : 0.8052


# Automated Machine learning (AutoML) 
In this section, Azure AutoML is used to automatically identify the best machine learning model and preprocessing pipeline for the Bank Marketing dataset.

In [44]:
from azureml.core import Dataset 
from azureml.train.automl import AutoMLConfig 

import pandas as pd 

In [45]:
df = pd.read_csv("bank.csv") 
print(df.shape) 
df.head() 

(11162, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


In [48]:
from azureml.core import Datastore 

# Get the default datastore 
datastore = ws.get_default_datastore() 

# Upload the local CSV 
datastore.upload(
    src_dir = ".",
    target_path = "bank-data",
    overwrite=True,
    show_progress = True
)

"Datastore.upload" is deprecated after version 1.0.69. Please use "Dataset.File.upload_directory" to upload your files             from a local directory and create FileDataset in single method call. See Dataset API change notice at https://aka.ms/dataset-deprecation.


Uploading an estimated of 15 files
Uploading ./.amlignore
Uploaded ./.amlignore, 1 files out of an estimated total of 15
Uploading ./.amlignore.amltmp
Uploaded ./.amlignore.amltmp, 2 files out of an estimated total of 15
Uploading ./bank.csv
Uploaded ./bank.csv, 3 files out of an estimated total of 15
Uploading ./config.json
Uploaded ./config.json, 4 files out of an estimated total of 15
Uploading ./train.py
Uploaded ./train.py, 5 files out of an estimated total of 15
Uploading ./train.py.amltmp
Uploaded ./train.py.amltmp, 6 files out of an estimated total of 15
Uploading ./Udacity-project.ipynb
Uploaded ./Udacity-project.ipynb, 7 files out of an estimated total of 15
Uploading ./udacity-project.ipynb.amltmp
Uploaded ./udacity-project.ipynb.amltmp, 8 files out of an estimated total of 15
Uploading ./.ipynb_aml_checkpoints/Udacity-project-checkpoint2026-6-9-13-46-46Z.ipynb
Uploaded ./.ipynb_aml_checkpoints/Udacity-project-checkpoint2026-6-9-13-46-46Z.ipynb, 9 files out of an estimated t

$AZUREML_DATAREFERENCE_f738a9032103445b9310ba60620c1a2c

In [49]:
from azureml.core import Dataset 

dataset = Dataset.Tabular.from_delimited_files(
    path = (datastore, "bank-data/bank.csv")
) 

print("Dataset created successfully.")

Dataset created successfully.


In [50]:
# Register the Dataset 

dataset = dataset.register(
    workspace = ws,
    name = "bank-dataset",
    create_new_version = True
) 

print(dataset.name)

bank-dataset


In [51]:
dataset.to_pandas_dataframe().head() 

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,False,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,False,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,False,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,False,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,False,unknown,yes


# Create the AutoML Configuration

In [56]:
from azureml.train.automl import AutoMLConfig 

automl_config = AutoMLConfig(
    task = "classification" , 
    training_data=dataset,
    label_column_name="deposit",
    primary_metric="accuracy",
    n_cross_validations=5,
    experiment_timeout_minutes = 30,
    enable_early_stopping=True,
    compute_target=compute_target,
   
)

print('AutoML configuration created successfully') 

AutoML configuration created successfully


# Automated Machine Learning (AutoML) Execution

The AutoML configuration was successfully created using the Azure Machine Learning SDK. However, during programmatic submission of the AutoML experiment (`experiment.submit(automl_config)`), an internal Azure ML SDK v1 environment issue prevented the experiment from being submitted successfully.

The issue was caused by version inconsistencies among the Azure ML SDK packages available in the managed notebook environment. Although the AutoML configuration was created successfully, the SDK raised an internal exception during dependency resolution while submitting the experiment.

To complete the project and evaluate the AutoML workflow, the same dataset, compute target, experiment, and training configuration were used to submit the AutoML experiment through the Azure Machine Learning Studio interface.

The following section describes each step performed in Azure Machine Learning Studio.


## AutoML Workflow in Azure Machine Learning Studio.

Step1: Open Azure Machine Learning Studio.
Navigate to the Azure Machine Learning Workspace and select **Automated ML** from the left navigation pane.

Step2: Create a New AutoML Job
Clicked **New Automated ML Job** to create a new AutoML experiment.

Step3: Select the Dataset
Select the previously registered dataset:
**Dataset Name:** `bank-datset` 
This dataset was created from the local `bank.csv` file using pandas and registered as an Azure ML Tabular Dataset.

Step4: Configure the Machine Learning Task 
Configure the AutoML task using the following settings:
- Task Type: **Classification** 
- Target Column: **deposit** 
- Validation Type: **Automatic** 
- Test Dataset: **None** 

Step5: Select the Compute Target 
Select the existing Azure Machine Learning Compute Cluster that was created during the HyperDrive implementation.
This allows both HyperDrive and AutoML experiments to execute on the same compute infrastructure. 

Step6: Configure Training Limits 
Configure the AutoML experiment using the following parameters:
- Primary Metric: **Accuracy** 
- Maximum Trials: **20** 
- Maximum Concurrent Trails: **4** 
- Maximum Compute Nodes: **1** 
- Experiment Timeout: **30 Minutes** 
- Iteration Timeout: **10 Minutes** 
- Early Termination: **Enabled**

Step7: Submit the AutoML Job 
Review the configuration and submit the AutoML experiment.
The experiment progresses through the following execution stages:
- Preparing
- Queued 
- Running
- Completed

Step8: Review the AutoML Results
After the experiment completed successfully, Azure Machine Learning evaluated multiple machine learning algorithms and preprocessing pipelines.
The Best-performing pipeline selected by AutoML was:
- **MaxAbsScaler + LightGBM** 
Primary Metric: - **Accuracy** 
Final Accuracy: - **0.85630** 

This Model achieved the highest accuracy among all the evaluated approaches in this project.









# AutoML Results Summary

The AutoML experiment successfully evaluated multiple machine learning algorithms and preprocessing pipelines.

Among all evaluated models, the **MaxAbsScaler + LightGBM** pipeline achieved the highest classification accuracy of **0.85630**. 

This Model outperformed both the baseline Logistic Regression model and the HyperDrive-optimized Logistic Regression model, demonstrating the effectiveness of Azure Automated Machine Learning in selecting an appropriate preprocessing pipeline and learning algorithm for this dataset.